In [21]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [22]:

df = pd.read_csv('Salary_Data.csv')
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


In [23]:
df.isnull().sum()

Age                    1
Gender                 1
Education Level        1
Job Title              1
Years of Experience    1
Salary                 1
dtype: int64

In [24]:
## drop rows where have null values
df.dropna(inplace=True)
df.isnull().sum()

Age                    0
Gender                 0
Education Level        0
Job Title              0
Years of Experience    0
Salary                 0
dtype: int64

In [25]:
cols_to_encode = ['Gender', 'Education Level', 'Job Title']
encoders = {}

for col in cols_to_encode:
    encoders[col] = LabelEncoder()
    df[col] = encoders[col].fit_transform(df[col])

df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,1,0,159,5.0,90000.0
1,28.0,0,1,17,3.0,65000.0
2,45.0,1,2,130,15.0,150000.0
3,36.0,0,0,101,7.0,60000.0
4,52.0,1,1,22,20.0,200000.0


In [26]:
X = df.drop(columns=['Salary'])
y = df['Salary'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape)

(297, 5) (297,)


In [27]:
scaler = StandardScaler()

cols_to_scale = ['Age', 'Years of Experience']

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

X_train.head()

,Age,Gender,Education Level,Job Title,Years of Experience
192,-0.507579,1,0,141,-0.482131
75,-0.077272,1,0,96,-0.016715
84,-1.224759,0,0,56,-1.257824
361,-0.651015,1,0,65,-0.792408
16,-0.651015,0,1,83,-0.482131


In [28]:

X_train_tensor = torch.from_numpy(X_train.values).float()
X_test_tensor  = torch.from_numpy(X_test.values).float()

# ✅ float32 not long (regression)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32)

In [29]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = features
        self.labels   = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


In [30]:
# Create dataset objects
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset  = CustomDataset(X_test_tensor,  y_test_tensor)

train_dataset[0]

(tensor([ -0.5076,   1.0000,   0.0000, 141.0000,  -0.4821]), tensor(95000.))

In [31]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [32]:
# Model
class MySimpleNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)    # ✅ 1 output, NO activation (regression)
        )

    def forward(self, features):
        return self.network(features)

In [33]:
epochs        = 100
learning_rate = 0.01

In [34]:
model     = MySimpleNN(num_features=X_train_tensor.shape[1])

loss_fn   = nn.MSELoss()    # ✅ MSE for regression

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [35]:
#tranning loop
for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:

        y_pred = model(batch_features)

        loss = loss_fn(y_pred.squeeze(), batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f'Epoch: {epoch + 1}, Loss: {avg_loss:.2f}')

Epoch: 1, Loss: 12202348748.80
Epoch: 2, Loss: 12115279257.60
Epoch: 3, Loss: 9678193152.00
Epoch: 4, Loss: 4889550860.80
Epoch: 5, Loss: 4108008115.20
Epoch: 6, Loss: 3690394124.80
Epoch: 7, Loss: 3504703654.40
Epoch: 8, Loss: 3527396185.60
Epoch: 9, Loss: 3370071129.60
Epoch: 10, Loss: 3381420480.00
Epoch: 11, Loss: 3443446643.20
Epoch: 12, Loss: 3204721331.20
Epoch: 13, Loss: 3240181593.60
Epoch: 14, Loss: 2911204428.80
Epoch: 15, Loss: 2592560646.40
Epoch: 16, Loss: 2603455193.60
Epoch: 17, Loss: 2095872294.40
Epoch: 18, Loss: 1459306694.40
Epoch: 19, Loss: 950260435.20
Epoch: 20, Loss: 598077028.80
Epoch: 21, Loss: 395161457.60
Epoch: 22, Loss: 389482147.20
Epoch: 23, Loss: 343248233.60
Epoch: 24, Loss: 334272848.00
Epoch: 25, Loss: 313837848.00
Epoch: 26, Loss: 308113228.80
Epoch: 27, Loss: 301424827.20
Epoch: 28, Loss: 280324923.20
Epoch: 29, Loss: 296078580.80
Epoch: 30, Loss: 294322024.00
Epoch: 31, Loss: 278597497.60
Epoch: 32, Loss: 294221804.80
Epoch: 33, Loss: 304721368.00

In [36]:
# Evaluation using MAE
model.eval()
predictions_list = []
actuals_list     = []

with torch.inference_mode():

    for batch_features, batch_labels in test_loader:

        y_pred = model(batch_features)
        predictions_list.extend(y_pred.squeeze().tolist())
        actuals_list.extend(batch_labels.tolist())

predictions = np.array(predictions_list)
actuals     = np.array(actuals_list)

mae = np.mean(np.abs(predictions - actuals))
print(f'MAE : ${mae:,.2f}')
print(f'Average Salary: ${actuals.mean():,.2f}')
print(f'Actual list: {actuals_list[:5]}')
print(f'Predicted list: {predictions_list[:5]}')

MAE : $11,067.34
Average Salary: $100,800.00
Actual list: [50000.0, 65000.0, 125000.0, 95000.0, 140000.0]
Predicted list: [44421.171875, 83150.8203125, 114165.4453125, 96880.640625, 149887.9375]


In [37]:
# ───────────────────────────────────────────────
# CELL 18 — Predict for new employee
# ───────────────────────────────────────────────

# Column order: Age, Gender, Education Level, Job Title, Years of Experience

# Gender          → Female=0,     Male=1
# Education Level → check encoder (alphabetical)
# Job Title       → check Cell 17 output above

new_employee = pd.DataFrame([{
    'Age': 30,
    'Gender': 1,
    'Education Level': 2,
    'Job Title': 50,
    'Years of Experience': 5,
}])

# scale Age and Years of Experience using the same feature names as during fit
new_employee[['Age', 'Years of Experience']] = scaler.transform(
    new_employee[['Age', 'Years of Experience']]
)

input_tensor = torch.from_numpy(new_employee.to_numpy(dtype=np.float32))


In [38]:
#prediction
model.eval()
with torch.inference_mode():
    output           = model(input_tensor)
    predicted_salary = output.item()    # ✅ directly float, no argmax, no threshold

print(f'Predicted Salary : ${predicted_salary:,.2f}')

Predicted Salary : $87,571.74
